In [1]:
import tensorflow as tf
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

In [2]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    "../data/color",
    batch_size=16,
    image_size=(224, 224),
    seed=42,
    subset="training",
    validation_split=0.25
)
test_dataset = tf.keras.utils.image_dataset_from_directory(
    "../data/color",
    batch_size=16,
    image_size=(224, 224),
    seed=42,
    subset="validation",
    validation_split=0.25
)
total = tf.data.experimental.cardinality(train_dataset).numpy()
train_size  = int(0.75 * total)

Found 54305 files belonging to 38 classes.
Using 40729 files for training.
Found 54305 files belonging to 38 classes.
Using 13576 files for validation.


In [3]:
class_names = train_dataset.class_names
print(class_names)

['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Sp

In [4]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.map(lambda x, y: (x / 255.0, y)).prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.map(lambda x, y: (x / 255.0, y)).prefetch(buffer_size=AUTOTUNE)

In [5]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2, fill_mode="nearest"),
    tf.keras.layers.RandomZoom(0.2, fill_mode="nearest"),
    tf.keras.layers.RandomContrast(0.2)
])

In [6]:
input = tf.keras.layers.Input(shape=(224, 224, 3))
x = data_augmentation(input)

In [7]:
for image, label in train_dataset.take(1):
    print("Image shape: ", image[0].shape)
    print("Label: ", label[0])


Image shape:  (224, 224, 3)
Label:  tf.Tensor(24, shape=(), dtype=int32)


In [8]:
y = np.concatenate([y for x, y in train_dataset], axis=0)
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y), y=y)
class_weights_dict = dict(enumerate(class_weights))

In [9]:
print(class_weights_dict)

{0: np.float64(2.261214745725072), 1: np.float64(2.3712738705170007), 2: np.float64(5.103884711779449), 3: np.float64(0.8699803485987696), 4: np.float64(0.9451638355147127), 5: np.float64(1.3883624215980366), 6: np.float64(1.6565931831123404), 7: np.float64(2.8735007760688585), 8: np.float64(1.184326839197441), 9: np.float64(1.430995713583023), 10: np.float64(1.2277385904624103), 11: np.float64(1.268421052631579), 12: np.float64(1.0256610425585495), 13: np.float64(1.3347643704529069), 14: np.float64(3.413426081126383), 15: np.float64(0.25851803894686065), 16: np.float64(0.6304798761609907), 17: np.float64(4.106573906029442), 18: np.float64(1.4702548552451087), 19: np.float64(0.9612697663441114), 20: np.float64(1.4386789120452137), 21: np.float64(1.4290877192982456), 22: np.float64(9.160818713450292), 23: np.float64(3.8693710811324338), 24: np.float64(0.2787557319827527), 25: np.float64(0.7800697157741515), 26: np.float64(1.2944635138571066), 27: np.float64(3.1067124332570555), 28: np.f

In [10]:
for images, labels in train_dataset.take(1):
    for num in labels:
        print(num.numpy())

10
25
24
1
20
21
32
35
37
33
24
28
29
16
25
15


In [11]:
SHUFFLE_BUFFER_SIZE = 1000
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.shuffle(SHUFFLE_BUFFER_SIZE).prefetch(buffer_size=AUTOTUNE).cache()
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE).cache()

In [ ]:
base_mode = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(256, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(len(class_names), activation="softmax")
])

: 

In [ ]:
real_model = tf.keras.models.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    data_augmentation,
    base_mode
])

real_model.compile(loss = tf.keras.losses.SparseCategoricalCrossentropy(),
              optimizer = tf.keras.optimizers.Adam(),
                metrics = ["accuracy"])
history = real_model.fit(train_dataset, epochs=10, validation_data=test_dataset, class_weight=class_weights_dict)

Epoch 1/10
